# [Chapter 10 — Sensitivity Analysis, §10.2] Tornado Plots for $\mathcal{R}_0$ and $I^*$

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 10 — Sensitivity Analysis
**Considerations developed:** 10
**Estimated runtime:** ≤ 30s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Generates tornado plots ranking parameters by their normalized sensitivity index. The plot for $\mathcal{R}_0$ has three +1 bars (c_I, beta, tau_R); the plot for $I^*$ has different rankings — visualizing the invasion-burden distinction.

## What you should already know
Chapter 10.1 local-sensitivity notebook.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_10
from shared.seeds import set_seed_chapter_10
from shared.verification import assert_within_tolerance
set_seed_chapter_10()
book_style()


In [ ]:
params = baseline_chapter_10()
c_I, beta, tau_R = params['c_I'], params['beta'], params['tau_R']
mu = 1/365
R0 = c_I * beta * tau_R

def R0_eval(p):
    return p['c_I'] * p['beta'] * p['tau_R']
def Istar_eval(p):
    R0_ = p['c_I'] * p['beta'] * p['tau_R']
    if R0_ <= 1: return 0
    return (mu / (mu + 1/p['tau_R'])) * (1 - 1/R0_)

def fd_indices(eval_func, base_params, param_names, h=1e-3):
    y0 = eval_func(base_params)
    indices = {}
    for name in param_names:
        p_plus = base_params.copy(); p_plus[name] = base_params[name] * (1 + h)
        p_minus = base_params.copy(); p_minus[name] = base_params[name] * (1 - h)
        indices[name] = (base_params[name]/y0) * (eval_func(p_plus) - eval_func(p_minus)) / (2 * h * base_params[name])
    return indices

names = ['c_I', 'beta', 'tau_R']
S_R0 = fd_indices(R0_eval, params, names)
S_Istar = fd_indices(Istar_eval, params, names)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, sens, title in zip(axes, [S_R0, S_Istar], ['Sensitivity of R_0', 'Sensitivity of I*']):
    keys = list(sens.keys())
    vals = list(sens.values())
    sorted_idx = np.argsort(np.abs(vals))
    keys_s = [keys[i] for i in sorted_idx]
    vals_s = [vals[i] for i in sorted_idx]
    colors = [BOOK_COLORS['positive'] if v >= 0 else BOOK_COLORS['negative'] for v in vals_s]
    ax.barh(keys_s, vals_s, color=colors, alpha=0.85)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel('Normalized sensitivity index')
    ax.set_title(title)
    ax.set_xlim(-1.2, 1.2)

plt.tight_layout()
plt.show()
print("\nTornado plots: R_0 has +1 sensitivity to all three parameters;")
print("I* sensitivities differ — this is the invasion-burden distinction quantified.")
